In [1]:
import pandas as pd
import time
import sqlite3
from nba_api.stats.endpoints import synergyplaytypes
from datetime import datetime


In [2]:
season = '2024-25'

In [3]:
playtypes = ['Cut','Handoff','Isolation','Misc','OffScreen','Postup','PRBallHandler','PRRollman','OffRebound','Spotup','Transition']

In [4]:
# List of play types to loop through
playtypes = ['Cut','Handoff','Isolation','Misc','OffScreen','Postup','PRBallHandler','PRRollman','OffRebound','Spotup','Transition']

# Initialize an empty list to hold DataFrames
all_data = []

# Loop through each play type and fetch data
for play_type in playtypes:
    try:
        # Fetch data for the given play type
        synergy_data = synergyplaytypes.SynergyPlayTypes(play_type_nullable=play_type,
                                                 season=season, 
                                                 season_type_all_star='Regular Season', 
                                                 per_mode_simple='Totals', 
                                                 player_or_team_abbreviation='P', 
                                                 type_grouping_nullable='offensive').get_data_frames()[0]
        
        
        # Append the fetched data to the list
        all_data.append(synergy_data)
        
        print(f"Fetched data for {play_type}")
    
    except Exception as e:
        print(f"Failed to fetch data for {play_type}: {e}")

# Combine all DataFrames into one
combined_df = pd.concat(all_data, ignore_index=True)

# Display the combined DataFrame
#print(combined_df.head())

Fetched data for Cut
Fetched data for Handoff
Fetched data for Isolation
Fetched data for Misc
Fetched data for OffScreen
Fetched data for Postup
Fetched data for PRBallHandler
Fetched data for PRRollman
Fetched data for OffRebound
Fetched data for Spotup
Fetched data for Transition


In [5]:
today = datetime.now().strftime('%Y-%m-%d')

In [6]:
combined_df['as_of'] = today

In [7]:
combined_df['ID'] = combined_df['as_of'].astype(str)+combined_df['PLAYER_ID'].astype(str)+combined_df['SEASON_ID'].astype(str)+combined_df['TEAM_ID'].astype(str)

In [8]:
combined_df

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,PLAY_TYPE,TYPE_GROUPING,PERCENTILE,GP,...,PLUSONE_POSS_PCT,SCORE_POSS_PCT,EFG_PCT,POSS,PTS,FGM,FGA,FGMX,as_of,ID
0,22024,1628386,Jarrett Allen,1610612739,CLE,Cleveland Cavaliers,Cut,Offensive,0.703,82,...,0.020,0.713,0.722,254,353,151,209,58,2025-04-20,2025-04-201628386220241610612739
1,22024,1627826,Ivica Zubac,1610612746,LAC,LA Clippers,Cut,Offensive,0.740,80,...,0.050,0.706,0.710,238,336,154,217,63,2025-04-20,2025-04-201627826220241610612746
2,22024,1630596,Evan Mobley,1610612739,CLE,Cleveland Cavaliers,Cut,Offensive,0.927,71,...,0.067,0.790,0.809,195,310,131,162,31,2025-04-20,2025-04-201630596220241610612739
3,22024,203497,Rudy Gobert,1610612750,MIN,Minnesota Timberwolves,Cut,Offensive,0.608,72,...,0.022,0.695,0.713,226,303,129,181,52,2025-04-20,2025-04-20203497220241610612750
4,22024,1628392,Isaiah Hartenstein,1610612760,OKC,Oklahoma City Thunder,Cut,Offensive,0.405,57,...,0.018,0.647,0.629,218,275,122,194,72,2025-04-20,2025-04-201628392220241610612760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3242,22024,1630544,Tre Mann,1610612766,CHA,Charlotte Hornets,Transition,Offensive,0.015,13,...,0.000,0.231,0.364,13,8,3,11,8,2025-04-20,2025-04-201630544220241610612766
3243,22024,1628470,Torrey Craig,1610612741,CHI,Chicago Bulls,Transition,Offensive,0.013,9,...,0.000,0.300,0.188,10,6,1,8,7,2025-04-20,2025-04-201628470220241610612741
3244,22024,1630692,Jordan Goodwin,1610612747,LAL,Los Angeles Lakers,Transition,Offensive,0.003,29,...,0.000,0.143,0.179,21,6,2,14,12,2025-04-20,2025-04-201630692220241610612747
3245,22024,203901,Elfrid Payton,1610612740,NOP,New Orleans Pelicans,Transition,Offensive,0.000,18,...,0.000,0.136,0.273,22,6,3,11,8,2025-04-20,2025-04-20203901220241610612740


In [9]:
#playertype_data.to_sql(table_name, conn, if_exists='replace', index=False)

In [10]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_type_offensive"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [11]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    combined_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = combined_df[~combined_df['ID'].isin(existing_ids['ID'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_type_offensive' already exists. Checking for new records...
Inserted 3247 new records into 'player_type_offensive'.


In [12]:
# List of play types to loop through
playtypes = ['Cut','Handoff','Isolation','Misc','OffScreen','Postup','PRBallHandler','PRRollman','OffRebound','Spotup','Transition']

# Initialize an empty list to hold DataFrames
all_data = []

# Loop through each play type and fetch data
for play_type in playtypes:
    try:
        # Fetch data for the given play type
        synergy_data = synergyplaytypes.SynergyPlayTypes(play_type_nullable=play_type,
                                                 season=season, 
                                                 season_type_all_star='Regular Season', 
                                                 per_mode_simple='Totals', 
                                                 player_or_team_abbreviation='T', 
                                                 type_grouping_nullable='defensive',
                                                        ).get_data_frames()[0]
        
        
        # Append the fetched data to the list
        all_data.append(synergy_data)
        
        print(f"Fetched data for {play_type}")
    
    except Exception as e:
        print(f"Failed to fetch data for {play_type}: {e}")

# Combine all DataFrames into one
combined_df = pd.concat(all_data, ignore_index=True)

# Display the combined DataFrame
#print(combined_df.head())

Fetched data for Cut
Fetched data for Handoff
Fetched data for Isolation
Fetched data for Misc
Fetched data for OffScreen
Fetched data for Postup
Fetched data for PRBallHandler
Fetched data for PRRollman
Fetched data for OffRebound
Fetched data for Spotup
Fetched data for Transition


In [13]:
combined_df['as_of'] = today
combined_df['ID'] = combined_df['as_of'].astype(str)+combined_df['SEASON_ID'].astype(str)+combined_df['TEAM_ID'].astype(str)

In [14]:
#teamtype_data.to_sql(table_name, conn, if_exists='replace', index=False)
combined_df

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,PLAY_TYPE,TYPE_GROUPING,PERCENTILE,GP,POSS_PCT,PPP,...,PLUSONE_POSS_PCT,SCORE_POSS_PCT,EFG_PCT,POSS,PTS,FGM,FGA,FGMX,as_of,ID
0,22024,1610612738,BOS,Boston Celtics,Cut,Defensive,0.655,82,0.059,1.279,...,0.027,0.656,0.658,523,669,291,442,151,2025-04-20,2025-04-20220241610612738
1,22024,1610612753,ORL,Orlando Magic,Cut,Defensive,0.690,82,0.061,1.277,...,0.036,0.653,0.668,531,678,288,431,143,2025-04-20,2025-04-20220241610612753
2,22024,1610612761,TOR,Toronto Raptors,Cut,Defensive,0.345,82,0.058,1.313,...,0.041,0.664,0.671,536,704,302,450,148,2025-04-20,2025-04-20220241610612761
3,22024,1610612752,NYK,New York Knicks,Cut,Defensive,0.000,82,0.059,1.376,...,0.036,0.693,0.700,534,735,312,446,134,2025-04-20,2025-04-20220241610612752
4,22024,1610612745,HOU,Houston Rockets,Cut,Defensive,0.897,82,0.065,1.240,...,0.035,0.629,0.650,596,739,316,486,170,2025-04-20,2025-04-20220241610612745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325,22024,1610612763,MEM,Memphis Grizzlies,Transition,Defensive,0.517,82,0.194,1.136,...,0.032,0.516,0.625,1863,2116,792,1439,647,2025-04-20,2025-04-20220241610612763
326,22024,1610612751,BKN,Brooklyn Nets,Transition,Defensive,0.172,82,0.207,1.156,...,0.034,0.519,0.631,1839,2126,787,1428,641,2025-04-20,2025-04-20220241610612751
327,22024,1610612764,WAS,Washington Wizards,Transition,Defensive,0.103,82,0.193,1.163,...,0.028,0.521,0.614,1829,2127,789,1481,692,2025-04-20,2025-04-20220241610612764
328,22024,1610612740,NOP,New Orleans Pelicans,Transition,Defensive,0.000,82,0.198,1.185,...,0.027,0.540,0.636,1836,2175,804,1439,635,2025-04-20,2025-04-20220241610612740


In [15]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "team_type_defensive"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [16]:
#combined_df['as_of'] = "2024-04-14"

In [17]:
#playertype_data['ID'] = playertype_data['as_of'].astype(str)+playertype_data['PLAYER_ID'].astype(str)+playertype_data['SEASON_ID'].astype(str)+playertype_data['TEAM_ID'].astype(str)

In [18]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    combined_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = combined_df[~combined_df['ID'].isin(existing_ids['ID'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'team_type_defensive' already exists. Checking for new records...
Inserted 330 new records into 'team_type_defensive'.


In [19]:
from nba_api.stats.endpoints import CommonAllPlayers
import pandas as pd

# Get all players for the 2023-24 season
players_data = CommonAllPlayers(is_only_current_season=1, league_id='00', season=season)
players_df = players_data.get_data_frames()[0]

# Extract relevant player IDs and names
player_ids = players_df['PERSON_ID'].tolist()
#print(f"Total players for 2023-24 season: {len(player_ids)}")


In [20]:
#player_ids

In [21]:
from nba_api.stats.endpoints import CommonPlayerInfo
import time
all_player_data = []
# Loop through each player ID to fetch shot data for the entire season
for player_id in player_ids:  # Using [:10] as a subset for testing
    attempt = 0
    success = False
    
    while attempt < 3 and not success:
        try:
            # Fetch shot chart details for the player across the entire season
            player_info = CommonPlayerInfo(player_id=player_id)
            # Convert the data to a DataFrame
            player_df = player_info.get_data_frames()[0]
            
            # Append the DataFrame to the list
            all_player_data.append(player_df)
            success = True
            
            # Respect API rate limits by adding a delay
            time.sleep(2)
        
        except Exception as e:
            attempt += 1
            print(f"Attempt {attempt} failed for player {player_id}: {e}")
            time.sleep(5* attempt)

# Combine all shot details into one DataFrame
#all_shots_df = pd.concat(all_shot_data, ignore_index=True)
all_player_data_df = pd.concat(all_player_data, ignore_index=True)


In [22]:
all_player_data_df[['PERSON_ID','FIRST_NAME', 'LAST_NAME','HEIGHT',
       'WEIGHT', 'SEASON_EXP']]

,PERSON_ID,FIRST_NAME,LAST_NAME,HEIGHT,WEIGHT,SEASON_EXP
0,1630173,Precious,Achiuwa,6-8,243,4
1,203500,Steven,Adams,6-11,265,10
2,1628389,Bam,Adebayo,6-9,255,7
3,1630534,Ochai,Agbaji,6-5,215,2
4,1630583,Santi,Aldama,7-0,215,3
...,...,...,...,...,...,...
570,1629027,Trae,Young,6-1,164,6
571,1627826,Ivica,Zubac,7-0,240,8
572,1641783,Tristan,da Silva,6-8,217,0
573,1628427,Vlatko,Čančar,6-8,236,4


In [23]:
# Function to convert height from 'X-Y' format to inches
def height_to_inches(height):
    feet, inches = height.split('-')
    return int(feet) * 12 + int(inches)

# Apply the function to the HEIGHT column
all_player_data_df['HEIGHT_INCHES'] = all_player_data_df['HEIGHT'].apply(height_to_inches)

In [24]:
all_player_data_sum = all_player_data_df[['PERSON_ID','FIRST_NAME', 'LAST_NAME','HEIGHT_INCHES',
       'WEIGHT', 'SEASON_EXP']]

In [25]:
all_player_data_sum.loc[all_player_data_sum['WEIGHT']=='','WEIGHT']=206

In [26]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_info"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

In [27]:
all_player_data_sum.to_sql(table_name, conn, if_exists='replace', index=False)
print(f"Table '{table_name}' created and data inserted.")

Table 'player_info' created and data inserted.


In [28]:
# Step 2: Create the table if it doesn't exist
if not table_exists:
    all_player_data_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = all_player_data_sum[~all_player_data_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_info' already exists. Checking for new records...


DatabaseError: Execution failed on sql 'SELECT id FROM player_info': no such column: id

In [ ]:
failed_player_logs = [1630846,1627751,203084,1629628]

26401

Table 'gamelogs' created and data inserted.
